Генерация инструкций и создание датасета для оценки IR метрик

In [16]:
!pip install -q openai
!pip install -q openai tqdm

In [17]:
from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import json
import time
from datetime import datetime
from collections import Counter
from tqdm.auto import tqdm
from openai import OpenAI
from pprint import pprint
import glob
import hashlib



client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)


In [ ]:
INPUT_PATH = "/content/drive/MyDrive/rusmsmarco_20k/rusmsmarco_20k_train.jsonl"


PREVIOUS_OUTPUTS = [
    "/content/drive/MyDrive/rusmsmarco_20k/openrouter_generated_1000.jsonl",
    "/content/drive/MyDrive/rusmsmarco_20k/openrouter_generated_2000_styles.jsonl",
    "/content/drive/MyDrive/rusmsmarco_20k/openrouter_generated_7000_styles.jsonl",
    "/content/drive/MyDrive/rusmsmarco_20k/openrouter_generated_7000_final.jsonl",
    "/content/drive/MyDrive/rusmsmarco_20k/openrouter_generated_extra_2000.jsonl",
]


CHUNK_DIR = "/content/drive/MyDrive/rusmsmarco_20k/generated_chunks"
os.makedirs(CHUNK_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_PATH = f"{CHUNK_DIR}/openrouter_chunk_{timestamp}.jsonl"

MODEL = "openrouter/owl-alpha"

N_TO_GENERATE = 100
BATCH_SIZE = 10

MAX_POS_CHARS = 800
MAX_NEG_CHARS = 400
MAX_TOKENS = 7000

SLEEP_BETWEEN_BATCHES = 8

STYLE_BLOCKS = [
    ("short", 20),
    ("background", 20),
    ("negation", 20),
    ("persona", 20),
    ("disambiguation", 20),
]

STYLE_REQUIREMENTS = {
    "short": [
        "Instruction должна быть короткой: одно предложение.",
        "Instruction должна лаконично задавать критерий релевантности.",
    ],
    "background": [
        "Instruction должна содержать краткий пользовательский контекст или background.",
        "Пример: 'Я готовлю справку о ...; релевантны документы, которые ...'",
    ],
    "negation": [
        "Instruction должна явно описывать, какие документы нерелевантны.",
        "Используй отрицательные ограничения: 'Нерелевантны документы, которые ...'",
    ],
    "persona": [
        "Instruction должна быть сформулирована от лица пользователя с ролью или задачей.",
        "Пример: 'Я аналитик/студент/врач/юрист и мне нужны документы, которые ...'",
    ],
    "disambiguation": [
        "Instruction должна устранять неоднозначность запроса.",
        "Укажи, какой смысл термина или сущности релевантен, а какие похожие смыслы нерелевантны.",
    ],
}

BAD_INSTRUCTION_PREFIXES = (
    "найдите",
    "найди",
    "ищите",
    "нужно найти",
)

BANNED_PHRASES = [
    "это не связано",
    "не соответствует",
    "не относится",
    "не помогает найти",
]


def list_chunk_files(chunk_dir):
    if not os.path.exists(chunk_dir):
        return []

    return [
        os.path.join(chunk_dir, x)
        for x in os.listdir(chunk_dir)
        if x.endswith(".jsonl")
    ]


def load_done_ids(paths):
    done = set()

    for path in paths:
        if not os.path.exists(path):
            continue

        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    qid = rec.get("query_id")
                    if qid is not None:
                        done.add(str(qid))
                except Exception:
                    continue

    return done


def make_batches(items, batch_size):
    batches = []

    for i in range(0, len(items), batch_size):
        batch_items = items[i:i + batch_size]
        style = batch_items[0]["instruction_style"]

        batches.append({
            "batch_idx": len(batches),
            "style": style,
            "items": batch_items,
        })

    return batches


def build_prompt(batch_items, style):
    compact_items = []

    for item in batch_items:
        compact_items.append({
            "id": item["id"],
            "query": item["query"],
            "positive": item["positive"],
            "negatives": item["negatives"],
        })

    return {
        "task": "generate_russian_instruction_retrieval_training_data",
        "language": "ru",
        "instruction_style": style,
        "style_requirements": STYLE_REQUIREMENTS[style],
        "requirements": [
            "Для каждого item сгенерируй одну instruction на русском языке.",
            "Instruction должна описывать критерий релевантности документов.",
            "Instruction должна быть такой, чтобы given positive passage оставался релевантным.",
            "Не добавляй требований, которых нет или почти нет в positive passage.",
            "Instruction НЕ должна начинаться с: Найдите, Найди, Ищите, Нужно найти.",
            "Instruction не должна раскрывать фактический ответ, если он не содержится явно в query.",
            "Сгенерируй ровно 3 hard negatives на русском языке.",
            "Каждый hard negative должен быть похож на исходный query, но НЕ соответствовать instruction.",
            "Hard negatives должны выглядеть как реальные поисковые passages.",
            "Hard negatives не должны копировать positive passage.",
            "Hard negatives не должны объяснять, почему они нерелевантны.",
            "Запрещены фразы: 'это не связано', 'не соответствует', 'не относится', 'не помогает найти'.",
            "Каждый hard negative должен быть самостоятельным passage на 35-60 слов.",
            "Не добавляй markdown, комментарии или текст вне JSON."
        ],
        "output_schema": {
            "items": [
                {
                    "id": "same id as input",
                    "instruction": "string, 1-2 Russian sentences unless style=short",
                    "hard_negatives": [
                        "string, 35-60 Russian words",
                        "string, 35-60 Russian words",
                        "string, 35-60 Russian words"
                    ]
                }
            ]
        },
        "items": compact_items
    }


def call_openrouter_for_batch(batch, max_retries=5):
    batch_idx = batch["batch_idx"]
    batch_items = batch["items"]
    style = batch["style"]
    expected_ids = [str(x["id"]) for x in batch_items]
    prompt = build_prompt(batch_items, style)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "Ты создаешь обучающие данные для русскоязычного dense retriever. "
                            "Отвечай только валидным JSON. Не используй markdown."
                        )
                    },
                    {
                        "role": "user",
                        "content": json.dumps(prompt, ensure_ascii=False)
                    }
                ],
                temperature=0.3,
                response_format={"type": "json_object"},
                max_tokens=MAX_TOKENS,
                extra_headers={
                    "HTTP-Referer": "https://colab.research.google.com/",
                    "X-Title": "rusmsmarco-chunk-generation"
                }
            )

            raw = response.choices[0].message.content

            if raw is None:
                raise ValueError("empty_response_content")

            result = json.loads(raw)

            return {
                "batch_idx": batch_idx,
                "style": style,
                "expected_ids": expected_ids,
                "result": result,
                "error": None,
            }

        except Exception as e:
            wait = min(120, 15 * (attempt + 1))
            print(f"Batch {batch_idx} style {style} attempt {attempt + 1} failed: {repr(e)}")
            print(f"Sleeping {wait}s")
            time.sleep(wait)

    return {
        "batch_idx": batch_idx,
        "style": style,
        "expected_ids": expected_ids,
        "result": None,
        "error": "failed_after_retries",
    }


def validate_item(gen, style):
    errors = []

    instruction = gen.get("instruction", "")

    if not isinstance(instruction, str) or len(instruction.strip()) < 20:
        errors.append("bad_instruction")

    if isinstance(instruction, str) and instruction.strip().lower().startswith(BAD_INSTRUCTION_PREFIXES):
        errors.append("bad_instruction_prefix")

    if style == "short" and isinstance(instruction, str):
        if len(instruction.split()) > 35:
            errors.append("short_instruction_too_long")

    hard_negatives = gen.get("hard_negatives")

    if not isinstance(hard_negatives, list) or len(hard_negatives) != 3:
        errors.append("bad_hard_negatives_count")
        return errors

    for j, text in enumerate(hard_negatives):
        if not isinstance(text, str):
            errors.append(f"hn{j}_not_string")
            continue

        word_count = len(text.split())
        lower = text.lower()

        if word_count < 25:
            errors.append(f"hn{j}_too_short:{word_count}")

        if word_count > 90:
            errors.append(f"hn{j}_too_long:{word_count}")

        if any(p in lower for p in BANNED_PHRASES):
            errors.append(f"hn{j}_contains_explanation_phrase")

    return errors



all_previous = PREVIOUS_OUTPUTS + list_chunk_files(CHUNK_DIR)
done_ids = load_done_ids(all_previous)
print("Already generated / will skip:", len(done_ids))
print("Chunk output:", OUTPUT_PATH)



style_queue = []
for style, count in STYLE_BLOCKS:
    style_queue.extend([style] * count)

if len(style_queue) != N_TO_GENERATE:
    raise ValueError("STYLE_BLOCKS sum must equal N_TO_GENERATE")


items = []
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        qid = str(item["query_id"])

        if qid in done_ids:
            continue

        if len(item.get("positive_passages", [])) < 1:
            continue

        if len(item.get("negative_passages", [])) < 2:
            continue

        style = style_queue[len(items)]

        items.append({
            "id": qid,
            "query": item["query"],
            "positive": item["positive_passages"][0]["text"][:MAX_POS_CHARS],
            "negatives": [
                item["negative_passages"][0]["text"][:MAX_NEG_CHARS],
                item["negative_passages"][1]["text"][:MAX_NEG_CHARS],
            ],
            "instruction_style": style,
            "source_item": {
                "query_id": qid,
                "query": item["query"],
                "positive_passages": item["positive_passages"][:1],
                "negative_passages": item["negative_passages"][:2],
            }
        })

        if len(items) >= N_TO_GENERATE:
            break

print("Items to generate:", len(items))
print("Styles:", Counter(x["instruction_style"] for x in items))
print("First id:", items[0]["id"] if items else None)
print("Last id:", items[-1]["id"] if items else None)

if len(items) == 0:
    raise ValueError("No new items to generate.")

batches = make_batches(items, BATCH_SIZE)
print("Batches:", len(batches))

source_by_id = {item["id"]: item["source_item"] for item in items}
style_by_id = {item["id"]: item["instruction_style"] for item in items}



start_time = time.time()
written = 0
failed_batches = 0
validation_errors = []

with open(OUTPUT_PATH, "w", encoding="utf-8") as fout:
    for batch in tqdm(batches, desc="Generating chunk 100"):
        rec = call_openrouter_for_batch(batch)

        if rec["result"] is None:
            failed_batches += 1
            print("Failed batch:", rec["batch_idx"], rec["style"])
            continue

        generated_items = rec["result"].get("items", [])

        for gen in generated_items:
            qid = str(gen.get("id"))
            source = source_by_id.get(qid)
            style = style_by_id.get(qid)

            if not source or not style:
                continue

            errors = validate_item(gen, style)

            out = {
                "query_id": qid,
                "query": source["query"],
                "positive_passages": source["positive_passages"],
                "negative_passages": source["negative_passages"],
                "only_instruction": gen.get("instruction", ""),
                "instruction_style": style,
                "generated_hard_negatives": [
                    {
                        "docid": f"{qid}_gen_{i}",
                        "title": "",
                        "text": text
                    }
                    for i, text in enumerate(gen.get("hard_negatives", []))
                    if isinstance(text, str)
                ],
                "generation_model": MODEL,
                "validation_errors": errors,
            }

            fout.write(json.dumps(out, ensure_ascii=False) + "\n")
            fout.flush()
            written += 1

            if errors:
                validation_errors.extend([f"{qid}:{e}" for e in errors])

        time.sleep(SLEEP_BETWEEN_BATCHES)

elapsed = time.time() - start_time

print("Saved chunk:", OUTPUT_PATH)
print("Written items:", written)
print("Failed batches:", failed_batches)
print("Elapsed min:", elapsed / 60)
print("Validation errors:", len(validation_errors))
print(validation_errors[:50])

rows = 0
bad_json = 0
empty_instruction = 0
not_enough_negs = 0
validation_error_rows = 0
styles = Counter()

with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        try:
            item = json.loads(line)
        except Exception:
            bad_json += 1
            continue

        rows += 1
        styles[item.get("instruction_style", "unknown")] += 1

        if not item.get("only_instruction", "").strip():
            empty_instruction += 1

        if len(item.get("generated_hard_negatives", [])) < 3:
            not_enough_negs += 1

        if item.get("validation_errors"):
            validation_error_rows += 1

print("Rows:", rows)
print("Bad JSON:", bad_json)
print("Empty instruction:", empty_instruction)
print("Not enough generated negatives:", not_enough_negs)
print("Rows with validation errors:", validation_error_rows)
print("Styles:", styles)

Already generated / will skip: 4243
Chunk output: /content/drive/MyDrive/rusmsmarco_20k/generated_chunks/openrouter_chunk_20260528_130809.jsonl
Items to generate: 100
Styles: Counter({'short': 20, 'background': 20, 'negation': 20, 'persona': 20, 'disambiguation': 20})
First id: 247812
Last id: 25614
Batches: 10


Generating chunk 100:   0%|          | 0/10 [00:00<?, ?it/s]

Saved chunk: /content/drive/MyDrive/rusmsmarco_20k/generated_chunks/openrouter_chunk_20260528_130809.jsonl
Written items: 100
Failed batches: 0
Elapsed min: 31.87534914414088
Validation errors: 7
['24921:bad_hard_negatives_count', '25000:hn1_too_short:22', '2505:bad_hard_negatives_count', '253271:hn2_too_short:23', '255114:hn2_too_short:24', '255359:bad_hard_negatives_count', '255700:hn2_too_short:24']
Rows: 100
Bad JSON: 0
Empty instruction: 0
Not enough generated negatives: 3
Rows with validation errors: 7
Styles: Counter({'short': 20, 'background': 20, 'negation': 20, 'persona': 20, 'disambiguation': 20})


Генерация инструкций для оценки IR метрик

In [20]:
INPUT_PATH = "/content/drive/MyDrive/rusmsmarco_20k/chunks_testset.jsonl"

CHUNK_DIR = "/content/drive/MyDrive/rusmsmarco_20k/metric_instruction_chunks"
os.makedirs(CHUNK_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_PATH = f"{CHUNK_DIR}/metric_instr_chunk_{timestamp}.jsonl"

MODEL = "openrouter/owl-alpha"

N_TO_GENERATE = 100
BATCH_SIZE = 10
MAX_TOKENS = 9000
SLEEP_BETWEEN_BATCHES = 8

MAX_QUERY_CHARS = 500
MAX_INSTR_CHARS = 900
MAX_PASSAGE_CHARS = 900

BAD_INSTRUCTION_PREFIXES = (
    "найдите",
    "найди",
    "ищите",
    "нужно найти",
)

ALLOWED_DOC_IDS = [
    "positive",
    "original_neg_0",
    "original_neg_1",
    "generated_neg_0",
    "generated_neg_1",
    "generated_neg_2",
]


def sha1_text(t: str) -> str:
    return hashlib.sha1(t.strip().encode("utf-8")).hexdigest()


def list_chunk_files(chunk_dir):
    if not os.path.exists(chunk_dir):
        return []
    return [
        os.path.join(chunk_dir, x)
        for x in os.listdir(chunk_dir)
        if x.endswith(".jsonl")
    ]


def load_done_ids(paths):
    done = set()

    for path in paths:
        if not os.path.exists(path):
            continue

        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    qid = rec.get("query_id")
                    if qid is not None:
                        done.add(str(qid))
                except Exception:
                    continue

    return done


def make_candidate_docs(item):
    return [
        {
            "doc_id": "positive",
            "text": item["positive"][:MAX_PASSAGE_CHARS],
            "current_role": "positive under only_instruction",
        },
        {
            "doc_id": "original_neg_0",
            "text": item["original_negs"][0][:MAX_PASSAGE_CHARS],
            "current_role": "negative",
        },
        {
            "doc_id": "original_neg_1",
            "text": item["original_negs"][1][:MAX_PASSAGE_CHARS],
            "current_role": "negative",
        },
        {
            "doc_id": "generated_neg_0",
            "text": item["generated_negs"][0][:MAX_PASSAGE_CHARS],
            "current_role": "negative",
        },
        {
            "doc_id": "generated_neg_1",
            "text": item["generated_negs"][1][:MAX_PASSAGE_CHARS],
            "current_role": "negative",
        },
        {
            "doc_id": "generated_neg_2",
            "text": item["generated_negs"][2][:MAX_PASSAGE_CHARS],
            "current_role": "negative",
        },
    ]


def build_prompt(batch_items):
    compact_items = []

    for item in batch_items:
        compact_items.append({
            "id": item["query_id"],
            "query": item["query"][:MAX_QUERY_CHARS],
            "only_instruction": item["only_instruction"][:MAX_INSTR_CHARS],
            "instruction_style": item.get("instruction_style"),
            "candidate_docs": make_candidate_docs(item),
        })

    return {
        "task": "generate_metric_instructions_for_russian_instruction_retrieval_eval",
        "language": "ru",
        "goal": (
            "Для каждого item нужно сгенерировать дополнительные инструкции для оценки "
            "instruction-following dense retriever: reverse_instruction для WISE/SICR "
            "и pmrr_instruction + pmrr_changed_docs для p-MRR."
        ),
        "requirements": [
            "Не изменяй query и only_instruction.",
            "reverse_instruction должна быть противоположной only_instruction.",
            "При reverse_instruction документ positive должен стать нерелевантным или явно менее релевантным.",
            "reverse_instruction не должна быть бессмысленной; она должна задавать реалистичный альтернативный критерий релевантности.",
            "pmrr_instruction должна быть modified/refined instruction: более узкой или измененной версией only_instruction.",
            "pmrr_instruction должна делать хотя бы один ранее релевантный документ нерелевантным.",
            "В нашем наборе positive считается ранее релевантным под only_instruction, поэтому обычно включай 'positive' в pmrr_changed_docs.",
            "pmrr_changed_docs — список doc_id из candidate_docs, которые были релевантны или потенциально релевантны под only_instruction, но должны стать нерелевантны под pmrr_instruction.",
            "Не включай в pmrr_changed_docs документы, для которых нет понятной причины стать нерелевантными.",
            "pmrr_changed_docs должен содержать минимум 1 doc_id и максимум 3 doc_id.",
            "Старайся, чтобы pmrr_instruction не совпадала с reverse_instruction: reverse — противоположность, pmrr — уточнение или сужение.",
            "Инструкции должны быть на русском языке.",
            "Инструкции не должны начинаться с: Найдите, Найди, Ищите, Нужно найти.",
            "Не раскрывай фактический ответ, если он не содержится явно в query.",
            "Не добавляй markdown, комментарии или текст вне JSON."
        ],
        "output_schema": {
            "items": [
                {
                    "id": "same id as input",
                    "reverse_instruction": "string, 1-2 Russian sentences",
                    "pmrr_instruction": "string, 1-2 Russian sentences",
                    "pmrr_changed_docs": [
                        {
                            "doc_id": "one of: positive, original_neg_0, original_neg_1, generated_neg_0, generated_neg_1, generated_neg_2",
                            "reason": "short Russian explanation why this doc should drop under pmrr_instruction"
                        }
                    ]
                }
            ]
        },
        "items": compact_items,
    }


def make_batches(items, batch_size):
    batches = []
    for i in range(0, len(items), batch_size):
        batches.append({
            "batch_idx": len(batches),
            "items": items[i:i + batch_size],
        })
    return batches


def call_openrouter_for_batch(batch, max_retries=5):
    batch_idx = batch["batch_idx"]
    expected_ids = [str(x["query_id"]) for x in batch["items"]]
    prompt = build_prompt(batch["items"])

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "Ты создаешь данные для оценки русскоязычного instruction-following dense retriever. "
                            "Отвечай только валидным JSON. Не используй markdown."
                        ),
                    },
                    {
                        "role": "user",
                        "content": json.dumps(prompt, ensure_ascii=False),
                    },
                ],
                temperature=0.25,
                response_format={"type": "json_object"},
                max_tokens=MAX_TOKENS,
                extra_headers={
                    "HTTP-Referer": "https://colab.research.google.com/",
                    "X-Title": "rusmsmarco-metric-instruction-generation",
                },
            )

            raw = response.choices[0].message.content
            if raw is None:
                raise ValueError("empty_response_content")

            result = json.loads(raw)

            return {
                "batch_idx": batch_idx,
                "expected_ids": expected_ids,
                "result": result,
                "error": None,
            }

        except Exception as e:
            wait = min(120, 15 * (attempt + 1))
            print(f"Batch {batch_idx} attempt {attempt + 1} failed: {repr(e)}")
            print(f"Sleeping {wait}s")
            time.sleep(wait)

    return {
        "batch_idx": batch_idx,
        "expected_ids": expected_ids,
        "result": None,
        "error": "failed_after_retries",
    }


def validate_generated(gen):
    errors = []

    reverse_instruction = gen.get("reverse_instruction", "")
    pmrr_instruction = gen.get("pmrr_instruction", "")
    pmrr_changed_docs = gen.get("pmrr_changed_docs")

    for field_name, text in [
        ("reverse_instruction", reverse_instruction),
        ("pmrr_instruction", pmrr_instruction),
    ]:
        if not isinstance(text, str) or len(text.strip()) < 20:
            errors.append(f"bad_{field_name}")
        elif text.strip().lower().startswith(BAD_INSTRUCTION_PREFIXES):
            errors.append(f"bad_prefix_{field_name}")

    if isinstance(reverse_instruction, str) and isinstance(pmrr_instruction, str):
        if reverse_instruction.strip().lower() == pmrr_instruction.strip().lower():
            errors.append("reverse_equals_pmrr")

    if not isinstance(pmrr_changed_docs, list):
        errors.append("bad_pmrr_changed_docs_type")
        return errors

    if len(pmrr_changed_docs) < 1:
        errors.append("empty_pmrr_changed_docs")

    if len(pmrr_changed_docs) > 3:
        errors.append("too_many_pmrr_changed_docs")

    seen = set()
    for j, obj in enumerate(pmrr_changed_docs):
        if not isinstance(obj, dict):
            errors.append(f"pmrr_changed_doc_{j}_not_object")
            continue

        doc_id = obj.get("doc_id")
        reason = obj.get("reason", "")

        if doc_id not in ALLOWED_DOC_IDS:
            errors.append(f"bad_pmrr_doc_id:{doc_id}")

        if doc_id in seen:
            errors.append(f"duplicate_pmrr_doc_id:{doc_id}")

        seen.add(doc_id)

        if not isinstance(reason, str) or len(reason.strip()) < 10:
            errors.append(f"bad_pmrr_reason:{doc_id}")

    if "positive" not in seen:
        errors.append("positive_not_in_pmrr_changed_docs")

    return errors




done_ids = load_done_ids(list_chunk_files(CHUNK_DIR))

print("Already generated / will skip:", len(done_ids))
print("Chunk output:", OUTPUT_PATH)

items = []

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        qid = str(item.get("query_id", ""))

        if not qid or qid in done_ids:
            continue

        if not item.get("query", "").strip():
            continue

        if not item.get("only_instruction", "").strip():
            continue

        if not item.get("positive", "").strip():
            continue

        if len(item.get("original_negs", [])) < 2:
            continue

        if len(item.get("generated_negs", [])) < 3:
            continue

        items.append(item)

        if len(items) >= N_TO_GENERATE:
            break

print("Items to generate:", len(items))
print("Styles:", Counter(x.get("instruction_style", "unknown") for x in items))
print("First id:", items[0]["query_id"] if items else None)
print("Last id:", items[-1]["query_id"] if items else None)

if len(items) == 0:
    raise ValueError("No new items to generate.")

batches = make_batches(items, BATCH_SIZE)
print("Batches:", len(batches))

source_by_id = {str(item["query_id"]): item for item in items}
start_time = time.time()
written = 0
failed_batches = 0
validation_errors = []

with open(OUTPUT_PATH, "w", encoding="utf-8") as fout:
    for batch in tqdm(batches, desc="Generating metric instructions"):
        rec = call_openrouter_for_batch(batch)

        if rec["result"] is None:
            failed_batches += 1
            print("Failed batch:", rec["batch_idx"])
            continue

        generated_items = rec["result"].get("items", [])

        for gen in generated_items:
            qid = str(gen.get("id"))
            source = source_by_id.get(qid)

            if not source:
                continue

            errors = validate_generated(gen)

            changed_docs = []
            for obj in gen.get("pmrr_changed_docs", []):
                if not isinstance(obj, dict):
                    continue

                doc_id = obj.get("doc_id")
                if doc_id not in ALLOWED_DOC_IDS:
                    continue

                if doc_id == "positive":
                    text = source["positive"]
                elif doc_id.startswith("original_neg_"):
                    idx = int(doc_id.split("_")[-1])
                    text = source["original_negs"][idx]
                elif doc_id.startswith("generated_neg_"):
                    idx = int(doc_id.split("_")[-1])
                    text = source["generated_negs"][idx]
                else:
                    continue

                changed_docs.append({
                    "doc_id": doc_id,
                    "text_hash": sha1_text(text),
                    "text": text,
                    "reason": obj.get("reason", ""),
                })

            out = dict(source)
            out.update({
                "reverse_instruction": gen.get("reverse_instruction", ""),
                "pmrr_instruction": gen.get("pmrr_instruction", ""),
                "pmrr_changed_docs": changed_docs,
                "metric_generation_model": MODEL,
                "metric_validation_errors": errors,
            })

            fout.write(json.dumps(out, ensure_ascii=False) + "\n")
            fout.flush()
            written += 1

            if errors:
                validation_errors.extend([f"{qid}:{e}" for e in errors])

        time.sleep(SLEEP_BETWEEN_BATCHES)

elapsed = time.time() - start_time

print("Saved chunk:", OUTPUT_PATH)
print("Written items:", written)
print("Failed batches:", failed_batches)
print("Elapsed min:", elapsed / 60)
print("Validation errors:", len(validation_errors))
print(validation_errors[:50])

rows = 0
bad_json = 0
empty_reverse = 0
empty_pmrr = 0
empty_changed_docs = 0
validation_error_rows = 0
styles = Counter()

with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        try:
            item = json.loads(line)
        except Exception:
            bad_json += 1
            continue

        rows += 1
        styles[item.get("instruction_style", "unknown")] += 1

        if not item.get("reverse_instruction", "").strip():
            empty_reverse += 1

        if not item.get("pmrr_instruction", "").strip():
            empty_pmrr += 1

        if not item.get("pmrr_changed_docs"):
            empty_changed_docs += 1

        if item.get("metric_validation_errors"):
            validation_error_rows += 1

print("Rows:", rows)
print("Bad JSON:", bad_json)
print("Empty reverse:", empty_reverse)
print("Empty pmrr:", empty_pmrr)
print("Empty changed docs:", empty_changed_docs)
print("Rows with validation errors:", validation_error_rows)
print("Styles:", styles)

Already generated / will skip: 574
Chunk output: /content/drive/MyDrive/rusmsmarco_20k/metric_instruction_chunks/metric_instr_chunk_20260531_125538.jsonl
Items to generate: 0
Styles: Counter()
First id: None
Last id: None


ValueError: No new items to generate.